In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold, cross_validate
import statsmodels.api as sm
import statsmodels.formula.api as smf

Load Data

In [8]:
univData = pd.read_csv("../Data/Clean/univData.csv")
print(len(univData))

445


Tests

In [9]:
# Pooled
m_pooled = smf.ols("gradRate ~ spendRate", data = univData).fit()
print(m_pooled.summary())
print(m_pooled.params["spendRate"])

                            OLS Regression Results                            
Dep. Variable:               gradRate   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     11.00
Date:                Tue, 01 Sep 2026   Prob (F-statistic):           0.000985
Time:                        20:31:52   Log-Likelihood:                 211.99
No. Observations:                 445   AIC:                            -420.0
Df Residuals:                     443   BIC:                            -411.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.5991      0.013     45.167      0.0

In [10]:
m_additive = smf.ols("gradRate ~ spendRate + C(division)", data = univData).fit()
print(m_additive.summary())

                            OLS Regression Results                            
Dep. Variable:               gradRate   R-squared:                       0.319
Model:                            OLS   Adj. R-squared:                  0.311
Method:                 Least Squares   F-statistic:                     41.17
Date:                Tue, 01 Sep 2026   Prob (F-statistic):           1.00e-34
Time:                        20:31:52   Log-Likelihood:                 292.08
No. Observations:                 445   AIC:                            -572.2
Df Residuals:                     439   BIC:                            -547.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [11]:
m_interaction = smf.ols("gradRate ~ spendRate * C(division)", data = univData).fit()
print(m_interaction.summary())

                            OLS Regression Results                            
Dep. Variable:               gradRate   R-squared:                       0.335
Model:                            OLS   Adj. R-squared:                  0.321
Method:                 Least Squares   F-statistic:                     24.36
Date:                Tue, 01 Sep 2026   Prob (F-statistic):           8.62e-34
Time:                        20:31:52   Log-Likelihood:                 297.33
No. Observations:                 445   AIC:                            -574.7
Df Residuals:                     435   BIC:                            -533.7
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [12]:
print(sm.stats.anova_lm(m_additive, m_interaction))

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0     439.0  7.010848      0.0       NaN       NaN       NaN
1     435.0  6.847503      4.0  0.163346  2.594208  0.035989


Corrected sample: slopes differ across divisions (F = 2.59, p = 0.036), driven by D-III's null relationship

In [13]:
m_size = smf.ols("gradRate ~ spendRate + np.log(totalExp) + C(division)", data=univData).fit()
print(m_size.params['spendRate'], m_size.pvalues['spendRate'])

1.0545422727656464 6.878550362841687e-08


Controlling for institution size flips the spendRate sign (−0.89 → +1.05)

In [14]:
m_alt = smf.ols("gradRate ~ np.log(sportsExpenses) + np.log(totalExp) + C(division)", data=univData).fit()
print(m_alt.summary())

                            OLS Regression Results                            
Dep. Variable:               gradRate   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.565
Method:                 Least Squares   F-statistic:                     97.27
Date:                Tue, 01 Sep 2026   Prob (F-statistic):           2.24e-77
Time:                        20:31:52   Log-Likelihood:                 394.97
No. Observations:                 445   AIC:                            -775.9
Df Residuals:                     438   BIC:                            -747.3
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Levels specification confirms the flip is not a ratio artifact (log sportsExpenses +0.078, p < 0.001).

Machine Learning Models

In [20]:
features = pd.read_csv("../Data/Clean/features.csv")

num = ["spendRate", "expPerStudent", "ugTotal", "admRate", "testScore",
       "upPell", "instrShare", "adminShare"]
ind = ["testScoreMissing", "admRateMissing"]
cat = ["division"]

X = features[num + ind + cat].assign(
    expPerStudent=lambda d: np.log(d["expPerStudent"]),
    ugTotal=lambda d: np.log(d["ugTotal"]),
)
y = features["gradRate"]
w = features["cohortSize"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale",  StandardScaler())]), num),
    ("ind", "passthrough", ind),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
])

models = {
    "linear": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "randomForest": RandomForestRegressor(n_estimators=500, random_state=42),
    "gbm": GradientBoostingRegressor(random_state=42)
}

pipes = {name: Pipeline([("prep", preprocess), ("model", m)])
         for name, m in models.items()}

cv = RepeatedKFold(n_splits= 5, n_repeats = 5, random_state= 42)

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size = .20, random_state = 42, stratify = features["division"])

results = {}
for name, pipe in pipes.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=["neg_root_mean_squared_error", "r2"],
                         params={"model__sample_weight": w_train})
    results[name] = {
        "rmse_mean": -res["test_neg_root_mean_squared_error"].mean(),
        "rmse_sd":    res["test_neg_root_mean_squared_error"].std(),
        "r2_mean":    res["test_r2"].mean(),
        "r2_sd":      res["test_r2"].std(),
    }

cv_table = pd.DataFrame(results).T.round(3)
print(cv_table)

              rmse_mean  rmse_sd  r2_mean  r2_sd
linear            0.081    0.009    0.693  0.077
ridge             0.081    0.009    0.693  0.077
randomForest      0.083    0.008    0.682  0.074
gbm               0.080    0.008    0.706  0.061
